# LiLaH Cross-Dataset Evaluation
Evaluates all 4 fine-tuned BERT models on the LiLaH hate speech dataset.

**Models:** BERT-age, BERT-gender, hateBERT-age, hateBERT-gender

**Before running:**
1. Runtime → Change runtime type → GPU (T4)
2. Make sure `hate_speech_only.tsv` is in `My Drive/thesis/`
3. Run all cells top to bottom

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
LILAH_FILE  = '/content/drive/MyDrive/thesis/hate_speech_only.tsv'
BASE        = '/content/drive/MyDrive/thesis'
RESULTS_DIR = f'{BASE}/lilah_results'

MODELS = [
    {'name': 'BERT-age',        'task': 'age',    'dir': 'bert_age_model'},
    {'name': 'BERT-gender',     'task': 'gender', 'dir': 'bert_gender_model'},
    {'name': 'hateBERT-age',    'task': 'age',    'dir': 'bert_age_hate_model'},
    {'name': 'hateBERT-gender', 'task': 'gender', 'dir': 'bert_gender_hate_model'},
]
# ─────────────────────────────────────────────────────────────────────────
print('Config loaded.')
print(f'LiLaH file: {LILAH_FILE}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
import os, json, gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

os.makedirs(RESULTS_DIR, exist_ok=True)

AGE_MAP = {
    '0-25':'0-25', '26-35':'26-35', '36-65':'36-65', '66-':'66-',
    '18-24':'0-25', '25-34':'26-35', '35-49':'36-65',
    '50-64':'36-65', '65-xx':'66-', '65+':'66-'
}
GENDER_MAP = {'male':'M','female':'F','m':'M','f':'F','M':'M','F':'F'}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load LiLaH once
lilah_df = pd.read_csv(LILAH_FILE, sep='\t')
print(f'LiLaH loaded: {len(lilah_df)} rows | columns: {list(lilah_df.columns)}')

In [ ]:
def find_model_path(folder):
    """Use root if model.safetensors exists there, else use highest-numbered checkpoint."""
    if os.path.exists(os.path.join(folder, 'model.safetensors')):
        return folder
    checkpoints = sorted(
        [d for d in os.listdir(folder) if d.startswith('checkpoint-')],
        key=lambda x: int(x.split('-')[1])
    )
    if not checkpoints:
        raise FileNotFoundError(f'No model weights found in {folder}')
    best = os.path.join(folder, checkpoints[-1])
    print(f'  Using checkpoint: {checkpoints[-1]}')
    return best


def plot_confusion_matrix(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

print('Helpers defined.')

In [ ]:
summary = []

for m in MODELS:
    folder = os.path.join(BASE, m['dir'])
    task   = m['task']

    print(f"\n{'='*60}")
    print(f"  {m['name']}  |  task={task}")
    print(f"{'='*60}")

    # Find model weights
    model_path = find_model_path(folder)

    # Label mapping is always in the root folder
    with open(os.path.join(folder, 'label_mapping.json')) as f:
        mapping = json.load(f)
    label2id = mapping['label2id']
    id2label = {int(k): v for k, v in mapping['id2label'].items()}
    print(f'  Labels: {label2id}')

    # Load model + tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device).eval()

    # Prepare LiLaH for this task
    df = lilah_df.dropna(subset=['text', task]).copy()
    df['text'] = df['text'].astype(str)
    norm_map = AGE_MAP if task == 'age' else GENDER_MAP
    df[task]  = df[task].astype(str).str.strip().map(lambda x: norm_map.get(x, x))
    df        = df[df[task].isin(label2id)].copy()
    print(f'  Rows: {len(df)}')
    print(f'  Distribution: {df[task].value_counts().to_dict()}')

    # Batch predict
    all_preds, texts = [], df['text'].tolist()
    for i in range(0, len(texts), 64):
        enc = tokenizer(
            texts[i:i+64], truncation=True, padding=True,
            max_length=128, return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            all_preds.extend(
                torch.argmax(model(**enc).logits, dim=1).cpu().tolist()
            )

    pred_labels = [id2label[i] for i in all_preds]
    true_labels = df[task].tolist()

    # Print classification report
    print(f'\n  Classification Report:')
    print(classification_report(true_labels, pred_labels, zero_division=0))

    # Confusion matrix
    labels_order = sorted(label2id.keys())
    cm = confusion_matrix(true_labels, pred_labels, labels=labels_order)
    cm_path = f"{RESULTS_DIR}/{m['name'].replace(' ','_')}_cm.png"
    plot_confusion_matrix(cm, labels_order, f"{m['name']} on LiLaH", cm_path)

    # Save predictions
    out = df.copy()
    out['predicted'] = pred_labels
    csv_path = f"{RESULTS_DIR}/{m['name'].replace(' ','_')}_predictions.csv"
    out.to_csv(csv_path, index=False)
    print(f'  Saved -> {csv_path}')

    # Store summary
    report = classification_report(true_labels, pred_labels, output_dict=True, zero_division=0)
    row = {'model': m['name'], 'task': task,
           'macro_f1': round(report['macro avg']['f1-score'], 4),
           'accuracy': round(report['accuracy'], 4)}
    if task == 'age':
        for cls in ['0-25', '26-35', '36-65', '66-']:
            row[f'f1_{cls}'] = round(report.get(cls, {}).get('f1-score', 0), 4)
    else:
        row['f1_F'] = round(report.get('F', {}).get('f1-score', 0), 4)
        row['f1_M'] = round(report.get('M', {}).get('f1-score', 0), 4)
    summary.append(row)

    # Free GPU memory
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('\n\nAll models evaluated.')

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────────────────
print('\n' + '='*65)
print('FINAL RESULTS — LiLaH Cross-Dataset Evaluation')
print('='*65)

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# Save summary
summary_path = f'{RESULTS_DIR}/lilah_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'\nSummary saved to: {summary_path}')

# Pretty print age models
print('\n── Age models ──')
age_df = summary_df[summary_df['task'] == 'age'][['model','macro_f1','f1_0-25','f1_26-35','f1_36-65','f1_66-']]
print(age_df.to_string(index=False))

print('\n── Gender models ──')
gen_df = summary_df[summary_df['task'] == 'gender'][['model','macro_f1','f1_F','f1_M']]
print(gen_df.to_string(index=False))